# Silver Control Setup

This notebook creates the schemas and control tables used by the Silver notebooks.

| Object | Purpose |
| --- | --- |
| `tuklas_dev`.`03_silver` | Cleaned and conformed Silver tables (geography, DOT travelers, PSA economy, population) |
| `tuklas_dev`.`06_data_quality` | Data-quality results for every Silver and Gold check (Issue #5) |
| `03_silver`.`silver_run_log` | One row per Silver notebook execution |
| `06_data_quality`.`dq_results` | One row per data-quality check per run |

The control tables are used by:

* `01_silver_geography`
* later Silver notebooks (`02_silver_dot`, `03_silver_psa_economy`, `04_silver_population`)

**Idempotency:** this notebook only uses `CREATE ... IF NOT EXISTS`. It never drops, truncates or overwrites anything, so it is safe to rerun.

### Step 1 — Configuration

Schema names follow the layer numbering used by `01_ingestion` and `02_bronze`. The Silver and data-quality schemas are created if they do not exist yet. Existing schemas are left unchanged.

In [0]:
CATALOG = "tuklas_dev"
SILVER_SCHEMA = "03_silver"
DQ_SCHEMA = "06_data_quality"

def silver_table(table_name):
    """Fully qualified, backtick-quoted identifier for a Silver table."""
    return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{table_name}`"

def dq_table(table_name):
    """Fully qualified, backtick-quoted identifier for a data-quality table."""
    return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{table_name}`"

for schema in (SILVER_SCHEMA, DQ_SCHEMA):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{schema}`")
    print(f"✓ Schema `{CATALOG}`.`{schema}` created / verified.")

### Table 1 — `silver_run_log`

**Purpose:** one row per execution of a Silver notebook. The row is inserted as `STARTED` and updated at the end of the run.

| Column | Type | Description |
| --- | --- | --- |
| `run_id` | STRING | Unique run identifier, e.g. `silver_geo_20261007_101500_ab12cd34` |
| `notebook_name` | STRING | e.g. `01_silver_geography` |
| `started_at` | TIMESTAMP | Run start time |
| `completed_at` | TIMESTAMP | Run end time |
| `tables_written` | BIGINT | Silver tables written in this run |
| `rows_written` | BIGINT | Total rows written across those tables |
| `dq_checks_run` | BIGINT | Data-quality checks evaluated |
| `dq_block_failures` | BIGINT | Failed `BLOCK` checks (any value above 0 means nothing was written) |
| `dq_review_flags` | BIGINT | Failed `REVIEW` checks (run continues; listed for review) |
| `status` | STRING | `STARTED`, `SUCCESS`, `SUCCESS_WITH_REVIEW`, `FAILED` |
| `notes` | STRING | Free-text notes |

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table('silver_run_log')} (
    run_id             STRING,
    notebook_name      STRING,
    started_at         TIMESTAMP,
    completed_at       TIMESTAMP,
    tables_written     BIGINT,
    rows_written       BIGINT,
    dq_checks_run      BIGINT,
    dq_block_failures  BIGINT,
    dq_review_flags    BIGINT,
    status             STRING,
    notes              STRING
) USING DELTA
""")

run_log_df = spark.table(silver_table("silver_run_log"))
print(f"✓ Table {silver_table('silver_run_log')} created / verified. Rows: {run_log_df.count()}")
run_log_df.printSchema()

### Table 2 — `dq_results`

**Purpose:** one row per data-quality check per run, following the failure policy in `docs/data-quality-rules.md` (Issue #5).

| Severity | What happens when the check fails |
| --- | --- |
| `BLOCK` | The notebook writes its results here, writes **no** Silver tables, and raises an error so the job stops |
| `REVIEW` | The run continues; the failure is listed for review before Gold is published |
| `LOG` | Recorded only (known, documented conditions) |

| Column | Type | Description |
| --- | --- | --- |
| `run_id` | STRING | Run that evaluated the check |
| `notebook_name` | STRING | Notebook that evaluated the check |
| `check_id` | STRING | e.g. `DQ-GEO-01`, `DQ-DOT-03` |
| `table_name` | STRING | Table or input checked |
| `severity` | STRING | `BLOCK`, `REVIEW`, `LOG` |
| `status` | STRING | `PASS` or `FAIL` |
| `failed_rows` | BIGINT | Rows (or groups) that failed |
| `expected` | STRING | Expected result, as text |
| `observed` | STRING | Observed result, as text |
| `sample_keys` | STRING | Up to 20 failing keys |
| `checked_at` | TIMESTAMP | When the check ran |
| `notes` | STRING | Free-text notes |

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {dq_table('dq_results')} (
    run_id         STRING,
    notebook_name  STRING,
    check_id       STRING,
    table_name     STRING,
    severity       STRING,
    status         STRING,
    failed_rows    BIGINT,
    expected       STRING,
    observed       STRING,
    sample_keys    STRING,
    checked_at     TIMESTAMP,
    notes          STRING
) USING DELTA
""")

dq_df = spark.table(dq_table("dq_results"))
print(f"✓ Table {dq_table('dq_results')} created / verified. Rows: {dq_df.count()}")
dq_df.printSchema()

### Summary

In [0]:
control_tables = [
    (silver_table("silver_run_log"), "one row per Silver notebook execution"),
    (dq_table("dq_results"),         "one row per data-quality check per run"),
]
summary_rows = []
for full_name, purpose in control_tables:
    exists = spark.catalog.tableExists(full_name)
    row_count = spark.table(full_name).count() if exists else -1
    summary_rows.append((full_name, purpose, exists, row_count))
    print(f"{'✓' if exists else '✗'} {full_name:<45} | {purpose} | rows: {row_count}")

display(spark.createDataFrame(summary_rows, "table_name STRING, purpose STRING, exists BOOLEAN, row_count BIGINT"))
print("\nSilver control setup complete.")